# Exercise 10: Single-head and multi-head temporal attention

Tutorial 10 trained one temporal-attention model on the same PeMSD4 detector used in Tutorial 07. Every five-minute token contains flow, occupancy and speed, and the target is the detector's flow one hour after the input window ends. This exercise investigates two attention structures from the lecture: single-head and multi-head self-attention, with tuned Ridge and LSTM sequence references.

Tune Ridge, LSTM, single-head attention, and multi-head attention separately on validation data. Freeze one final model per family, then compare all of them with persistence on test data once.


## Open this notebook

- **Colab.** Open the notebook, then run the setup cell below before any data-loading cell. It clones the public course repository and changes into `10 Attention Mechanisms`, so relative data paths work.
- **On your own computer.** Open the notebook from its lesson folder so the data files sit beside it.


In [ ]:
from pathlib import Path
import os

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    print("Running in Google Colab")
    REPO_URL = "https://github.com/hku-kejintao/AI-Code.git"
    LESSON_DIR = "10 Attention Mechanisms"
    base = Path("/content")
    repo = base / "AI-Code"
    os.chdir(base)
    if not repo.exists():
        !git clone {REPO_URL}
    lesson_path = repo / LESSON_DIR
    if not lesson_path.exists():
        raise FileNotFoundError(f"Lesson folder not found: {lesson_path}")
    os.chdir(lesson_path)
else:
    print("Running on your own computer")

print("Current folder:", os.getcwd())


### Check the packages

The cell should print version numbers and the word `ready`. NumPy holds arrays, pandas holds tables, Matplotlib draws figures, and scikit-learn provides preprocessing, models, and metrics. PyTorch builds the neural networks used in this lesson.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("matplotlib", plt.matplotlib.__version__)
print("scikit-learn", sklearn.__version__)
print("torch", torch.__version__)
print("ready")


### Install, only if the check failed

Skip this cell when the check printed `ready`. After installation finishes, restart the kernel and run the cells above again.


In [ ]:
%pip install numpy pandas matplotlib scikit-learn torch


In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import display
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42); torch.manual_seed(42); torch.set_num_threads(2)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("device:",DEVICE)

In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self,input_size,lookback,d_model=32,n_heads=4,dropout=.1):
        super().__init__()
        self.input=nn.Linear(input_size,d_model)
        self.position=nn.Parameter(torch.randn(1,lookback,d_model)*.02)
        self.attention=nn.MultiheadAttention(d_model,n_heads,dropout=dropout,batch_first=True)
        self.norm1=nn.LayerNorm(d_model)
        self.ff=nn.Sequential(
            nn.Linear(d_model,2*d_model),nn.ReLU(),nn.Dropout(dropout),nn.Linear(2*d_model,d_model)
        )
        self.norm2=nn.LayerNorm(d_model)
        self.output=nn.Linear(d_model,1)

    def forward(self,x,return_attention=False):
        h=self.input(x)+self.position[:,:x.shape[1]]
        attended,weights=self.attention(h,h,h,need_weights=True,average_attn_weights=False)
        h=self.norm1(h+attended)
        h=self.norm2(h+self.ff(h))
        prediction=self.output(h[:,-1])
        return (prediction,weights) if return_attention else prediction


def fit(model,train_loader,valid_loader,epochs=8,lr=1e-3,weight_decay=0):
    optimizer=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=weight_decay)
    loss_fn=nn.MSELoss()
    history={"train MAE":[],"valid MAE":[]}
    best=(float("inf"),None)
    for epoch in range(epochs):
        model.train()
        total=0
        train_absolute_error=0
        train_targets=0
        for xb,yb in train_loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            optimizer.zero_grad()
            prediction=model(xb)
            loss=loss_fn(prediction,yb)
            loss.backward()
            optimizer.step()
            total+=loss.item()*len(xb)
            train_absolute_error+=torch.abs((prediction-yb)*flow_std).sum().item()
            train_targets+=yb.numel()
        model.eval()
        validation_total=0
        valid_absolute_error=0
        valid_targets=0
        with torch.no_grad():
            for xb,yb in valid_loader:
                xb,yb=xb.to(DEVICE),yb.to(DEVICE)
                prediction=model(xb)
                validation_total+=loss_fn(prediction,yb).item()*len(xb)
                valid_absolute_error+=torch.abs((prediction-yb)*flow_std).sum().item()
                valid_targets+=yb.numel()
        train_loss=total/len(train_loader.dataset)
        valid_loss=validation_total/len(valid_loader.dataset)
        train_mae=train_absolute_error/train_targets
        valid_mae=valid_absolute_error/valid_targets
        history["train MAE"].append(train_mae)
        history["valid MAE"].append(valid_mae)
        if valid_mae<best[0]:
            best=(valid_mae,copy.deepcopy(model.state_dict()))
        print(
            f"epoch {epoch+1:02d} | "
            f"train MAE {train_mae:.2f} | "
            f"validation MAE {valid_mae:.2f} vehicles/5 min"
        )
    model.load_state_dict(best[1])
    return history


def predict(model,X):
    model.eval()
    outputs=[]
    loader=DataLoader(TensorDataset(torch.tensor(X)),256)
    with torch.no_grad():
        for (xb,) in loader:
            outputs.append(model(xb.to(DEVICE)).cpu().numpy())
    return np.concatenate(outputs)


def score(name,prediction,target):
    error=prediction-target
    return {
        "model":name,
        "MAE":np.abs(error).mean(),
        "RMSE":np.sqrt((error**2).mean()),
    }


class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x):
        sequence, _ = self.lstm(x)
        return self.output(sequence[:, -1])


In [ ]:
traffic=pd.read_csv(
    "pems04_single_sensor.csv.gz",
    parse_dates=["time"],
).set_index("time")
traffic=traffic.iloc[:12000].copy()

values=traffic.to_numpy(dtype="float32")
train_end,valid_end=7500,9500
feature_mean=values[:train_end].mean(axis=0)
feature_std=values[:train_end].std(axis=0)
scaled_values=(values-feature_mean)/feature_std
flow_mean=float(feature_mean[0])
flow_std=float(feature_std[0])

def make_windows(lookback,forecast_horizon=12):
    input_windows=[]
    flow_targets=[]
    target_positions=[]
    first_target=lookback+forecast_horizon-1
    for target_position in range(first_target,len(scaled_values)):
        input_end=target_position-forecast_horizon
        input_start=input_end-lookback+1
        input_windows.append(scaled_values[input_start:input_end+1])
        flow_targets.append(scaled_values[target_position,0])
        target_positions.append(target_position)
    X=np.asarray(input_windows,dtype="float32")
    y=np.asarray(flow_targets,dtype="float32").reshape(-1,1)
    return X,y,np.asarray(target_positions)

LOOKBACK=48
X,y,target_pos=make_windows(LOOKBACK)
train=target_pos<train_end
valid=(target_pos>=train_end)&(target_pos<valid_end)
test=target_pos>=valid_end
X_train,y_train=X[train],y[train]
X_valid,y_valid=X[valid],y[valid]
X_test,y_test=X[test],y[test]
train_loader=DataLoader(
    TensorDataset(torch.tensor(X_train),torch.tensor(y_train)),
    batch_size=128,
    shuffle=True,
)
valid_loader=DataLoader(
    TensorDataset(torch.tensor(X_valid),torch.tensor(y_valid)),
    batch_size=256,
)
print("N =",len(X_train),"training samples")
print("T =",X_train.shape[1],"historical five-minute intervals")
print("D =",X_train.shape[2],"features: flow, occupancy, speed")
print("X_train:",X_train.shape,"y_train:",y_train.shape)
print("X_valid:",X_valid.shape,"y_valid:",y_valid.shape)
print("X_test :",X_test.shape,"y_test :",y_test.shape)
print("The shapes above are model arrays: N samples × T intervals × D traffic variables.")
first_window_raw=X_train[0]*feature_std+feature_mean
print("First input window in original units:")
display(pd.DataFrame(
    first_window_raw,
    columns=["flow [vehicles/5 min]","occupancy [fraction]","speed"],
).head(12).round(2))
print("Scaling is used only for model fitting; all reported prediction errors use flow units.")


## Question 1 — What do $Q$, $K$ and $V$ control?

Create a small `Q`, `K`, `V` example and compute the standard scaled dot-product attention used in the lecture:

$$
A=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right),\qquad O=AV.
$$

Run two controlled changes. First change only the first query vector and identify which row of the score, weight and output matrices changes. Then restore $Q$ and change only the third value vector; verify that the attention weights stay fixed while the retrieved output changes. Visualise the original and modified-query weight matrices and explain, in your own words, what $Q/K$ and $V$ control.


In [ ]:
# TODO: calculate the original scaled dot-product scores, weights and output.
# TODO: change only Q[0] and compare the rows of scores, weights and output.
# TODO: restore Q, change only V[2], and verify that weights do not change.
# TODO: visualise the original and modified-query weight matrices.


## Question 2 — Tune Ridge and LSTM sequence references

Evaluate persistence numerically and explicitly reshape the sequence from `(N, T, 3)` to the Ridge design matrix `(N, T × 3)` before tuning regularisation with validation MAE. The target has shape `(N, 1)` and represents flow one hour ahead. Tune Ridge alpha and LSTM hidden size separately; freeze one setting per family and report shapes, MAE/RMSE, and training time. Defer observed-versus-predicted curves to test data.


In [ ]:
# TODO: tune Ridge alpha and LSTM hidden size using validation MAE.
# TODO: print the Ridge (N, T x D) and LSTM (N, T, D) input shapes.


## Question 3 — Tune single-head and multi-head families

Tune model dimension and dropout separately for a single-head attention model and a multi-head attention model. Keep positional embeddings and the feed-forward block in both. Freeze one configuration per family using validation MAE and report parameter count and training time.


In [ ]:
# TODO: tune the supplied single-head and multi-head configurations on validation data.
# TODO: freeze one configuration per family.


## Question 4 — Visualize within-family validation tuning

Plot Ridge validation MAE against `alpha`, LSTM MAE against hidden size, and attention validation MAE against configuration separately for the single-head and multi-head families. Use these plots only to freeze settings. Defer attention-weight interpretation to held-out test examples.


In [ ]:
# TODO: plot within-family validation sensitivity for Ridge, LSTM and attention.
# Do not inspect test predictions in this step.


## Question 5 — One final Ridge, LSTM and attention comparison and attention interpretation

Evaluate persistence, frozen Ridge, frozen LSTM, frozen single-head attention, and frozen multi-head attention on test data once. Plot the held-out flow predictions and final MAE, then visualize final-query attention for a low-flow and a high-flow test example. Do not retune from these results.


In [ ]:
# TODO: evaluate persistence, Ridge, LSTM and both attention families on test once.
# TODO: plot a held-out sequence and interpret attention weights for two test cases.
